# **Decoder Casual Language Model**

In [1]:
import torchtext
import torch
import math
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Dataset**

In [2]:
train_iter,val_iter = IMDB()
next(iter(train_iter))

(1,
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

In [3]:
# define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

## Preprocessing Data

This section focuses on preparing text data for NLP tasks, mainly through tokenization and vocabulary creation.

### Special Tokens and Indices
Defines special tokens such as `<unk>`, `<pad>`, and an empty string (used as an EOS token), and assigns them indices (`0`, `1`, and `2`). These tokens serve specific purposes in text processing:

- `UNK_IDX`: Represents unknown or out-of-vocabulary words  
- `PAD_IDX`: Used to pad shorter sequences in a batch for uniform length  
- `EOS_IDX`: Indicates the end of a sequence  

### `yield_tokens` Function
A generator function that iterates through the dataset (`data_iter`), applies a tokenizer to each sample, and yields tokenized outputs one at a time.

### Building the Vocabulary
Constructs a vocabulary from the tokenized data using `build_vocab_from_iterator`. Special tokens are added at the beginning, and all tokens with a minimum frequency of 1 are included.

### Handling Unknown Tokens
Sets a default index (`UNK_IDX`) for tokens not found in the vocabulary, ensuring the model can handle unseen words gracefully.

### `text_to_index` Function
Converts raw text into a sequence of numerical indices based on the vocabulary, making it suitable as input for machine learning models.

### `index_to_en` Function
Transforms a sequence of indices back into readable text, which is useful for interpreting model outputs.

### Functionality Check
Validates the preprocessing pipeline by converting a sample tensor `[0, 1, 2]` back into the corresponding special tokens, ensuring that the mappings are working correctly.

In [4]:
UNK_IDX, PAD_IDX, EOS_IDX = 0, 1, 2
special_tokens = ["<unk>","<pad>", "<eos>" ]

In [5]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens(dataset):
    for id, text in dataset:
        yield tokenizer(text)
        
vocab = build_vocab_from_iterator(yield_tokens(train_iter),specials=special_tokens,
                                  special_first=True)
vocab.set_default_index(UNK_IDX)

In [6]:
vocab["store"]

1024

#### **Build Pipelines**

Text to Indices

In [7]:
def text_pipeline (sample):
    return [vocab[text] for text in tokenizer(sample) ]

text_pipeline("I rented I AM CURIOUS-YELLOW from my video store because of all")

[13, 1153, 13, 230, 24141, 50, 72, 351, 1024, 87, 9, 40]

Indices to Text

In [8]:
vocab.get_itos()[1153]

'rented'

In [9]:
def index_to_en (indices):
    tokens_list = [vocab.get_itos()[index] for index in indices]
    tokens_list = " ".join(tokens_list)
    return tokens_list
    
index_to_en([13, 1153, 13, 230, 24141, 50, 72, 351, 1024, 87, 9, 40])

'i rented i am curious-yellow from my video store because of all'

### Collate Function

For the decoder model, a collate function is designed to process batches of text data. This function takes a block of text as input and returns a transformed version suitable for training.

The transformation is handled by the `get_sample(block_size, text)` function. This function randomly extracts a segment of text (`src_sequence`) along with its shifted counterpart (`tgt_sequence`), which represents the next tokens in the sequence.

It ensures that the sampled text fits within the specified block size. If the original text is shorter than the block size, adjustments are made accordingly. The function ultimately returns both the source and target sequences, which are used as inputs for training the language model.

In [10]:
def get_sample(block_size, text):
    
    text_len = len(text)
    stop_point = 0
    start_point = 0
    seq_differ = text_len-block_size 
    
    if (seq_differ >= 1):
        start_point = torch.randint(low=0, high = seq_differ, size=(1,)).item() 
        stop_point = start_point + block_size
        
        src_sequence = text[start_point:stop_point]
        tgt_sequence = text[start_point+ 1: stop_point+1]
    
    elif (seq_differ <= 0):
        start_point = 0
        stop_point = text_len
        
        src_sequence = text[start_point:stop_point]
        tgt_sequence = text[start_point+1: stop_point]
        
        tgt_sequence.append('<|endoftext|>')
    
    return src_sequence, tgt_sequence
    

In [11]:
batch_size = 1

batch_of_tokens = []
for i in range(batch_size):
    _,text = next(iter(train_iter))
    batch_of_tokens.append(tokenizer(text))
    

In [12]:
text = batch_of_tokens[0][0:100]
text[0:50]

['i',
 'rented',
 'i',
 'am',
 'curious-yellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 '.',
 'i',
 'also',
 'heard',
 'that',
 'at',
 'first',
 'it',
 'was',
 'seized',
 'by',
 'u',
 '.',
 's',
 '.',
 'customs',
 'if',
 'it',
 'ever',
 'tried',
 'to',
 'enter',
 'this',
 'country',
 ',',
 'therefore']

In [13]:
block_size = 10
src_seq, tgt_seq = get_sample(block_size, text)
src_seq,tgt_seq

(['see',
  'this',
  'for',
  'myself',
  '.',
  'the',
  'plot',
  'is',
  'centered',
  'around'],
 ['this',
  'for',
  'myself',
  '.',
  'the',
  'plot',
  'is',
  'centered',
  'around',
  'a'])

In [14]:
vocab(src_seq)

[77, 15, 20, 479, 3, 4, 101, 11, 5505, 191]

### **Define Collate Function**

In [15]:
from torch.nn.utils.rnn import pad_sequence

In [16]:
block_size = 30
def collate_func (batch):
    src_batch, tgt_batch = [], []
    
    for _,text in batch:
        src_sequence, tgt_sequence = get_sample(block_size, tokenizer(text))
        #define as token indices
        src_sequence = vocab(src_sequence)
        tgt_sequence = vocab(tgt_sequence)
        
        #define as tensors
        src_sequence = torch.tensor(src_sequence,dtype=torch.int64)
        tgt_sequence = torch.tensor(tgt_sequence, dtype=torch.int64)
        
        #append to batch lists
        src_batch.append(src_sequence)
        tgt_batch.append(tgt_sequence)
    
    
    src_batch = pad_sequence(src_batch,batch_first=False, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch,batch_first=False, padding_value=PAD_IDX)
    
    return src_batch.to(device), tgt_batch.to(device)
        

### **Define Dataloaders**

In [17]:
from torch.utils.data import DataLoader

In [18]:
train_dataloader = DataLoader(train_iter,batch_size=batch_size,
                              shuffle=True, collate_fn=collate_func)

val_dataloader = DataLoader(val_iter, batch_size=batch_size,
                            shuffle=True, collate_fn=collate_func)


### Iterating through data samples

The provided code iterates through batches of source-target pairs from a data loader. It demonstrates how to access and print a few samples from the dataset:

- We initialize an iterator over the data loader named `dataset`.
- A loop runs for 10 iterations to fetch and print the first 10 source-target pairs. For each pair:
    - `src` and `trt` (short for target) hold the batch of source and target sequences respectively.
    - The `index_to_en` function is used to convert these sequences from numerical indices back to readable text.
    - The `sample` number and corresponding source and target texts are printed out.

After printing the first 10 samples, the code continues to iterate through the dataset:

- It prints the shape of the target and source tensors for the next batch, which provides information about the number of tokens and batch size.
- The `index_to_en` function is again used to convert the first sequence of the batch from indices to text for both source and target.
- Only the first pair of the remaining batches is printed, and then the loop breaks.

This process is useful for verifying that the data loader is functioning correctly and that the sequences are being properly transformed.


In [19]:
dataset = iter(train_dataloader)
for sample in range(10):
    src, tgt = next(dataset)
    print("sample",sample)
    print("sorce:",index_to_en(src))
    print("\n")
    print("target:",index_to_en(tgt))
    print("\n")

sample 0
sorce: one of the worst major films of all time , preposterous and inexcusable on every level . it tries to be clever , but its conception of feminism seems hopelessly


target: of the worst major films of all time , preposterous and inexcusable on every level . it tries to be clever , but its conception of feminism seems hopelessly anachronistic


sample 1
sorce: that they are not animals or subhumans but actually equal to himself . the problem is the role of the pygmies in the film - two people who are kidnapped


target: they are not animals or subhumans but actually equal to himself . the problem is the role of the pygmies in the film - two people who are kidnapped ,


sample 2
sorce: just didn ' t put enough time in perfecting it . the stories were pretty cool and creepy enough , but it was lacking . it ' s a good


target: didn ' t put enough time in perfecting it . the stories were pretty cool and creepy enough , but it was lacking . it ' s a good movie


sample 3
sorce

In [20]:
for  src,trt in dataset:
    print(trt.shape)
    print(src.shape)
    print(index_to_en(src[0,:]))
    print(index_to_en(trt[0,:]))
    break

torch.Size([30, 1])
torch.Size([30, 1])
look
passable


### **Masking**

In transformers, masking is crucial for ensuring certain positions are not attended to. The function ```generate_square_subsequent_mask``` produces an upper triangular matrix, which ensures that during decoding, a token can't attend to future tokens of target.


In [21]:
def get_square_subsequent_mask(seq_len, device = device):
    mask = (torch.triu(torch.ones((seq_len,seq_len),device=device)) == 1).transpose(0,1)
    mask = mask.float().masked_fill(mask == 0,float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask    

#### create a mask

In [22]:
def create_mask (src, device=device):
    src_len = src.shape[0]
    src_mask = get_square_subsequent_mask(src_len,device)
    src_padding_mask = (src == PAD_IDX).transpose(0,1)
    return src_mask, src_padding_mask

#### Testing Mask Functions 

In [23]:
src[0:4] = PAD_IDX
src[0:4]

tensor([[1],
        [1],
        [1],
        [1]])

In [24]:
mask, padding_mask = create_mask(src)
print("src", src)



src tensor([[   1],
        [   1],
        [   1],
        [   1],
        [1011],
        [   5],
        [  45],
        [ 228],
        [ 527],
        [ 809],
        [  20],
        [   4],
        [ 218],
        [ 310],
        [   3],
        [ 359],
        [   5],
        [ 199],
        [ 580],
        [   8],
        [  24],
        [  85],
        [ 119],
        [  66],
        [  79],
        [  86],
        [  10],
        [   4],
        [5319],
        [   9]])


In [25]:
print("\nmask", mask)
mask.shape



mask tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 

torch.Size([30, 30])

In [26]:
print("\npadding mask", padding_mask)
padding_mask.shape


padding mask tensor([[ True,  True,  True,  True, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False]])


torch.Size([1, 30])

### **Positional Encoding**

The Transformer model does not inherently understand the order of tokens in a sequence. To provide this information, positional encodings are added to the token embeddings, following a predefined pattern based on each token’s position.

In contrast, GPT uses learnable (trainable) positional encodings. Instead of fixed patterns like the sinusoidal encodings introduced in the original Transformer, these encodings are treated as parameters that are updated during training.

Each position in the input sequence is assigned its own learnable vector, with the same dimensionality as the token embeddings. As training progresses, the model adjusts these vectors along with its other parameters, enabling it to better capture positional relationships.

This approach allows GPT to develop more flexible and task-specific representations of position, which can enhance performance across different natural language processing tasks.

For simplicity, however, this lab uses fixed positional encodings.

In [27]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, seq_len, d_model, dropout):
        super(PositionalEncoding,self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        positions = torch.arange(0,seq_len).float().unsqueeze(1)
        
        pe = torch.zeros((seq_len,d_model))
        
        div_term = torch.exp(
            torch.arange(0,d_model,2) * (-math.log(10000)/d_model)
        )
        
        pe[:,0::2] = torch.sin(positions * div_term)
        pe[:,1::2] = torch.cos(positions * div_term)
        
        pe = pe.unsqueeze(1)
        
        self.register_buffer("pe", pe)
        
    def forward(self, word_embedding):
        len_word_embed = word_embedding.size(0)
        
        positional_embeddings = word_embedding + self.pe[0:len_word_embed,:, :]
        
        return self.dropout(positional_embeddings)

### **Token Embedding**

Token embedding (also called word embedding or word representation) is a technique used to map words or tokens from a text corpus into numerical vectors within a continuous vector space. Each unique token is represented by a fixed-length vector, where the values capture linguistic characteristics such as meaning, context, and relationships with other words.

The TokenEmbedding class below transforms numerical token indices into their corresponding embedding vectors.

#### **When you define nn.Embedding(vocab_size, d_model), PyTorch creates a learnable embedding matrix of size:**

vocab_size×d_model
Each row corresponds to a word/token in the vocabulary
Each row is a vector of size d_model representing that token

This matrix doesn’t initially “contain meaning” — it starts with random values and learns meaningful representations during training.

What happens in forward()

When you pass seq_tokens (which are token indices):

The embedding layer looks up the corresponding rows from the embedding matrix
So for a sequence of length N, you get an output of shape:
N×d_model

That means:

Each token in the sequence is converted into its embedding vector
All these vectors together form the embedding matrix for that sequence

In [28]:
class TokenEmbedding(nn.Module):
    
    def __init__(self,vocab_size,d_model):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size,d_model)
        self.d_model = d_model
        
    def forward(self, tokens):
        embed_out = self.embeddings(tokens.long())
        return embed_out * math.sqrt(self.d_model)
        

## Custom GPT model architecture

The `CustomGPTModel` class defines a transformer-based model architecture for generative pre-trained models. This model aims to generate text and perform various NLP tasks. Below is an explanation of the main components of the class:

- **Initialization (`__init__`)**: The constructor takes several parameters including `embed_size`, `vocab_size`, `num_heads`, `num_layers`, `max_seq_len`, and `dropout`. It initializes the embedding layer, positional encoding, transformer encoder layers, and a linear layer (`lm_head`) for generating logits over the vocabulary.

- **Weight initialization (`init_weights`)**: This method initializes the weights of the model for better training convergence. The Xavier uniform initialization is used, which is a common practice for initializing weights in deep learning.

- **Decoder (`decoder`)**: Although named `decoder`, this method currently functions as the forward pass through the transformer encoder layers, followed by the generation of logits for the language modeling task. It handles the addition of positional encodings to the embeddings and applies a mask if necessary.

- **Forward pass (`forward`)**: This method is similar to the `decoder` method and defines the forward computation of the model. It processes the input through embedding layers, positional encoding, transformer encoder layers, and produces the final output using the `lm_head`.

- **Mask generation**: Both `decoder` and `forward` methods contain logic to generate a square causal mask if no source mask is provided. This mask ensures that the prediction for a position does not depend on the future tokens in the sequence, which is important for the autoregressive nature of GPT models.

- **Commented out decoder**: A section of the code is commented out, suggesting an initial design where a transformer decoder layer was considered. However, the final implementation uses only encoder layers, which is a common simplification for models focusing on language modeling and generation.

This class effectively encapsulates the necessary components to create a GPT-like model, allowing for training on language modeling tasks and text generation applications.


In [29]:
class CutomGPTModel(nn.Module):
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len, dropout ):
        
        super(CutomGPTModel,self).__init__()
        self.init_weight()
        self.embeddings = TokenEmbedding(vocab_size,d_model)
        self.position_embeddings = PositionalEncoding(max_seq_len, d_model, dropout)
        
        self.encoder_transformer_layer = nn.TransformerEncoderLayer(d_model, n_heads,dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer=self.encoder_transformer_layer, num_layers=n_layers)
        self.d_model = d_model
        
        self.fc = nn.Linear(d_model, vocab_size)
    
    def init_weight(self):
        
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
                
    def create_mask(self, src, device = device):
        src = src.to(device)
        src_seq_len = src.shape[0]
    
        src_mask = nn.Transformer.generate_square_subsequent_mask(src_seq_len).to(device)
        src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    
        return src_mask, src_padding_mask
    
    def decoder(self, seq_tokens, src_mask = None):
        
        #define scaled word embeddings
        embed_out = self.embeddings(seq_tokens) * math.sqrt(self.d_model)
        #print("embed shape: ", embed_out.shape)
        
        #define positional embeddings
        positional_embeddings = self.position_embeddings(embed_out)
        #print("pos shape: ", positional_embeddings.shape)
        
        if src_mask is None:
            # if pre define mask is none, then create a new mask
            src_mask, src_pad_mask = self.create_mask(seq_tokens)
        else:
            src_pad_mask = None
        
        #print("src mask shape: ", src_mask.shape)
        #print("src pad shape: ", src_pad_mask.shape)
        output = self.transformer_encoder(positional_embeddings, src_mask,
                                          src_key_padding_mask=src_pad_mask)
        logits = self.fc(output)
        
        return logits
    
    def forward(self, seq_tokens, src_mask=None, seq_pad_mask=None):
        
        embed_out = self.embeddings(seq_tokens) * math.sqrt(self.d_model)
        
        postional_embeddings = self.position_embeddings(embed_out)
        
        if src_mask is None:
            
            src_mask, src_padding_mask = self.create_mask(seq_tokens)
        
        output = self.transformer_encoder(postional_embeddings, src_mask, src_padding_mask)
        logits = self.fc(output)
        
        return logits
        

#### Initiate the Model

In [30]:
vocab_size = len(vocab)
d_model = 200
n_layers = 2
n_heads = 2
dropout = 0.1
max_seq_len = 500

In [31]:
customize_GPT_model =  CutomGPTModel(vocab_size,d_model,n_heads,n_layers,max_seq_len, dropout)

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


### **Prompting**

We just need to ensure our text generate model will get the input in very accurate manner. So we use prompting to prompt the user inputs to the model align with requirments.

* The prompt cannot be empty.
* The prompt cannot exceed the limit of the block size.
* The prompt mustbe tokenize and convert into token indices.

In [32]:
def prompting (prompt, device = device):
    
    if prompt is None:
        prompt = input("Please enter your thoughts...(prompt cannot be empty)")
    
    prompt_tokens = tokenizer(prompt)
    prompt_len = len(prompt_tokens)
    
    if prompt_len > block_size:
        
        # You cut it down to fit the model by keeping the most recent tokens:
        prompt_tokens = prompt_tokens[-block_size:]
    
    #convert tokens into indices
    prompt_indices = vocab(prompt_tokens)
    prompt_indices = torch.tensor(prompt_indices, dtype=torch.int64).unsqueeze(1)
    
    
    return prompt_indices.to(device)

### **More Important!!!**

What does -block_size mean?

In Python, negative indexing means “count from the end of the list.”

So:

tokens[0] → first element
tokens[-1] → last element
tokens[-2] → second last
tokens[-block_size:] → last block_size elements

Why do we do this?

* Because your model (like a Transformer/GPT) has a maximum context length (block_size).

* You cut it down to fit the model by keeping the most recent tokens:

Why keep the last tokens?

In language models:

* The latest words matter most for prediction
* Earlier context becomes less important

#### Testing

In [33]:
text = "i rented i am curious-yellow from my video "
tokens = tokenizer(text)
tokens

indices = vocab(tokens)
indices = torch.tensor(indices,dtype=torch.int64)
indices,indices.shape

(tensor([   13,  1153,    13,   230, 24141,    50,    72,   351]),
 torch.Size([8]))

In [34]:
prmpt_indices = prompting(text)
prmpt_indices, prmpt_indices.size()

(tensor([[   13],
         [ 1153],
         [   13],
         [  230],
         [24141],
         [   50],
         [   72],
         [  351]]),
 torch.Size([8, 1]))

In [35]:
logits = customize_GPT_model.decoder(prmpt_indices)
logits,logits.shape

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\torch\nn\functional.py:5076: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


(tensor([[[-7.8032e-01,  1.9889e-01, -9.4717e-01,  ...,  9.2368e-01,
           -6.8208e-01,  2.5479e-01]],
 
         [[-1.3463e+00, -3.6072e-01,  4.2668e-01,  ...,  1.0416e+00,
           -5.8739e-01,  6.1433e-01]],
 
         [[-1.3647e+00, -8.7305e-02, -6.5130e-01,  ...,  8.9229e-01,
           -4.3587e-01,  5.3696e-02]],
 
         ...,
 
         [[ 8.4824e-04, -2.8506e-01, -4.6224e-01,  ...,  1.9174e-01,
            6.5059e-01,  5.6894e-01]],
 
         [[ 4.2876e-01, -8.8027e-01, -8.6618e-02,  ...,  3.2723e-01,
           -7.5187e-01, -2.0162e-01]],
 
         [[-7.0668e-01, -7.3790e-01, -1.7352e+00,  ...,  1.2635e+00,
            3.9821e-01, -7.5473e-02]]], grad_fn=<ViewBackward0>),
 torch.Size([8, 1, 68813]))

In [36]:
logits_shape = logits.transpose(0,1)
logits_shape.shape  

torch.Size([1, 8, 68813])

In [37]:
logits_shape

tensor([[[-7.8032e-01,  1.9889e-01, -9.4717e-01,  ...,  9.2368e-01,
          -6.8208e-01,  2.5479e-01],
         [-1.3463e+00, -3.6072e-01,  4.2668e-01,  ...,  1.0416e+00,
          -5.8739e-01,  6.1433e-01],
         [-1.3647e+00, -8.7305e-02, -6.5130e-01,  ...,  8.9229e-01,
          -4.3587e-01,  5.3696e-02],
         ...,
         [ 8.4824e-04, -2.8506e-01, -4.6224e-01,  ...,  1.9174e-01,
           6.5059e-01,  5.6894e-01],
         [ 4.2876e-01, -8.8027e-01, -8.6618e-02,  ...,  3.2723e-01,
          -7.5187e-01, -2.0162e-01],
         [-7.0668e-01, -7.3790e-01, -1.7352e+00,  ...,  1.2635e+00,
           3.9821e-01, -7.5473e-02]]], grad_fn=<TransposeBackward0>)

In [38]:
logit_pred = logits_shape[:,-1]
logit_pred,logit_pred.shape

(tensor([[-0.7067, -0.7379, -1.7352,  ...,  1.2635,  0.3982, -0.0755]],
        grad_fn=<SelectBackward0>),
 torch.Size([1, 68813]))

In [39]:
# get index of next word
max_index = torch.argmax(logit_pred,dim=1)

In [40]:
index_to_en(max_index)

'recollections'

### **Auto-Regressive Text Generation**

In decoder-based models, we generate text by repeatedly feeding the model’s output back into its input, extending the sequence step by step. This process continues until the model produces the end-of-sequence token <|endoftext|> or the sequence reaches a predefined length limit. We will later implement this as a function in the notebook.

In [41]:
prompt = "this is the beginning of"

prompt_indices = prompting(prompt).to(device)
print(f"promt indices: {prompt_indices}, prompt indices shape: {prompt_indices.shape}")

promt indices: tensor([[ 15],
        [ 11],
        [  4],
        [480],
        [  9]]), prompt indices shape: torch.Size([5, 1])


In [42]:
max_new_tokens = 10

In [43]:

for i in  range(max_new_tokens):
    
    logits = customize_GPT_model.decoder(prompt_indices,src_mask=None)
    print("logit shape:", logits.shape)
    
    logit_pred = logits.transpose(0,1)
    print(f"logit pred shape: {logit_pred.shape}")
    
    # get the last predict token as the generate token
    logit_pred = logit_pred[:,-1]
    logit_pred_token = torch.argmax(logit_pred, dim=-1).unsqueeze(1)
    
    #append the generate token for our previous prompt indices
    prompt_indices = torch.cat((prompt_indices,logit_pred_token),dim=0).to(device)
    print(f"sequence step {i}: {[index_to_en(j) for j in prompt_indices]}")
    print(f"Shape of prompt_encoded after concatenation at step {i}: {prompt_indices.shape}")
    
    
    
    

logit shape: torch.Size([5, 1, 68813])
logit pred shape: torch.Size([1, 5, 68813])
sequence step 0: ['this', 'is', 'the', 'beginning', 'of', 'cliches']
Shape of prompt_encoded after concatenation at step 0: torch.Size([6, 1])
logit shape: torch.Size([6, 1, 68813])
logit pred shape: torch.Size([1, 6, 68813])
sequence step 1: ['this', 'is', 'the', 'beginning', 'of', 'cliches', 'spelt']
Shape of prompt_encoded after concatenation at step 1: torch.Size([7, 1])
logit shape: torch.Size([7, 1, 68813])
logit pred shape: torch.Size([1, 7, 68813])
sequence step 2: ['this', 'is', 'the', 'beginning', 'of', 'cliches', 'spelt', 'disorder']
Shape of prompt_encoded after concatenation at step 2: torch.Size([8, 1])
logit shape: torch.Size([8, 1, 68813])
logit pred shape: torch.Size([1, 8, 68813])
sequence step 3: ['this', 'is', 'the', 'beginning', 'of', 'cliches', 'spelt', 'disorder', 'vests']
Shape of prompt_encoded after concatenation at step 3: torch.Size([9, 1])
logit shape: torch.Size([9, 1, 68813

### **Text Generate Function**

In [44]:
## Lets make it as a function

def generate(prompt, model, vocab, block_size,  max_token_len = 500, device = device):
    
    model = model.to(device)
    
    # encode the prompt and get the prompt indices
    prompt_indices = prompting(prompt)
    tokens = []
    
    for i in range(max_token_len):
        
        logits = model.decoder(prompt_indices,src_mask=None).to(device)
        logits = logits.transpose(0,1)
        
        last_pred_token = logits[:,-1]
        
        pred_token = torch.argmax(last_pred_token, dim=1)
        
        # If the next token is the end-of-sequence (EOS) token, stop generation
        if pred_token.item() == EOS_IDX:
            break
        
        #reshape it as [seq_len, batch_size] to cat with the previous prompt indices
        pred_token = pred_token.unsqueeze(1)
        
        #combine previous prompt indices and pred_token
        prompt_indices = torch.cat((prompt_indices,pred_token), dim=0)[-block_size:]
        
        # Convert the next token index to a token string using the vocabulary
        # Move the tensor back to CPU for vocab lookup if needed
        token_id =  pred_token.to('cpu').item()
        tokens.append(vocab.get_itos()[token_id])
        
    return " ".join(tokens)

In [45]:
generate(prompt,customize_GPT_model,vocab,20,20)

'unmarysuish joker attachés supremes martians travel--they francesco scarecrow-ized capitalized combinatoric theological meretricious skinny-dipping sleezebag mock-shock masculin scolded bushi nautical safe-zone'

### **Training VS Inference**

The key difference between training and inference lies in the decoder inputs. During training, the decoder uses teacher forcing, where the ground truth target sequence (shifted by one position) is provided as input. This allows the model to process the entire sequence in parallel while a causal mask ensures that each position only attends to previous tokens.

During inference, the model no longer has access to the ground truth and must generate tokens sequentially, using its own previous predictions as input.

In [46]:
src, tgt = next(iter(train_dataloader))
src

tensor([[    3],
        [   15],
        [11271],
        [  584],
        [  150],
        [    8],
        [   24],
        [ 2721],
        [  306],
        [   85],
        [    5],
        [   21],
        [   44],
        [   32],
        [   42],
        [  135],
        [ 3484],
        [  561],
        [    4],
        [   25],
        [   16],
        [ 5416],
        [    4],
        [ 1059],
        [    9],
        [    4],
        [   25],
        [   69],
        [   73],
        [  235]])

In [47]:
src_mask, src_pad_mask = create_mask(src)

In [48]:
logits = customize_GPT_model.decoder(src, src_mask)
logits, logits.shape

(tensor([[[ 0.1574,  0.5583, -0.7614,  ...,  0.8260,  0.1426,  0.3146]],
 
         [[ 0.0206,  0.6591, -0.2193,  ...,  1.1640, -0.4532,  0.3741]],
 
         [[ 0.3269,  0.1766, -0.1459,  ...,  1.2601, -0.0278, -0.6209]],
 
         ...,
 
         [[-0.3588, -0.0981, -0.1375,  ...,  1.2532,  0.7018,  0.2179]],
 
         [[-0.7737, -0.3915, -0.1897,  ...,  0.6387,  0.1847, -0.1378]],
 
         [[-0.2504, -0.4923, -0.6143,  ...,  0.6849, -0.1316, -0.1735]]],
        grad_fn=<ViewBackward0>),
 torch.Size([30, 1, 68813]))

In [49]:
# we need to initiate loss function
loss_func = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

**reshape - flattens all dimensions into one**

In [50]:
# reshape the logit values
logits = logits.reshape(-1,logits.shape[-1])
logits.shape

torch.Size([30, 68813])

In [51]:
tgt.shape

torch.Size([30, 1])

In [52]:
tgt = tgt.reshape(-1)
tgt.shape

torch.Size([30])

In [53]:
loss = loss_func(logits,tgt)
print(loss.item())

11.441905975341797


### **Define Evaluation Function** 

In [54]:
def evaluate (model: nn.Module, eval_data) -> float:
    
    model.eval()
    loss_val = 0
    
    with torch.no_grad():
        for src, tgt in eval_data:
            tgt = tgt.to(device)
            logits = model.decoder(src, src_mask = None)
            loss_val += loss_func(logits.reshape(-1,logits.shape[-1]), tgt.reshape(-1)).item()
            
            
    return loss_val / (len(list(eval_data))-1 )
            

In [55]:
evaluate(customize_GPT_model,val_dataloader)

11.347505552443778

### **Model Training**

##### Define Optimizer and Learning Rate Scheduler

In [62]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm

In [57]:
batch_size = 32
num_epochs = 3
dataset_size = len(list(train_dataloader))


In [58]:
total_steps = int(num_epochs * (dataset_size/batch_size))

In [67]:
optimizer = AdamW(customize_GPT_model.parameters(), lr = 1e-4)
lr_scheduler = StepLR(optimizer, 10000, gamma=0.9)

In [ ]:
def model_train (model: nn.Module, dataloader, optimizer, lr_scheduler, loss_func, epochs = 3):
    
    model.train()
    total_loss = []
    num_batch = len(list(dataloader)) // block_size
    
    for epoch in tqdm(range(epochs)):
        epoch_loss = 0
        
        for batch, src_tgt in enumerate(dataloader):
            
            optimizer.zero_grad()
            src = src_tgt[0]
            tgt = src_tgt[1]
            
            #get logit values
            logits = model(src, src_mask=None, src_pad_mask = None)
            logits_flattern = logits.reshape(-1, logits.shape[-1])
            
            loss = loss_func(logits_flattern, tgt.reshape(-1))
            #backpropagation and weights update
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),0.5)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        #update learning rate at each epoch
        lr_scheduler.step()
        
        total_loss.append(epoch_loss)
        
        
        
        
        
            